# 02 · Approximate Nearest Neighbors (ANN) & HNSW
Exact search compares the query to **every** vector — fine for 1000 docs, impossible for 10 million. **ANN** trades a tiny bit of accuracy for massive speed. **HNSW** is the ANN algorithm most vector databases (Chroma, Weaviate, pgvector) use.

## 1. The problem: exact search is O(N)
```
  exact (brute force):  compare query to ALL N vectors   -> slow at scale
  ANN:                  cleverly skip most of them        -> ~log(N), tiny accuracy loss
```

In [ ]:
import numpy as np, time
rng = np.random.default_rng(0)
N, dim = 50000, 64
data = rng.standard_normal((N, dim)).astype("float32")
query = rng.standard_normal(dim).astype("float32")

# exact nearest neighbor (brute force) — the ground truth
t=time.time()
dists = np.linalg.norm(data - query, axis=1)
exact_top = dists.argsort()[:5]
print("exact top-5:", exact_top, f"  ({(time.time()-t)*1000:.1f} ms over {N} vectors)")

## 2. HNSW — a navigable graph you 'walk' toward the answer
```
  HNSW = Hierarchical Navigable Small World graph

  layer 2 (sparse):   o---------o---------o      <- few long-range links (big jumps)
                       \         |        /
  layer 1:            o--o--o--o--o--o--o--o     <- medium links
                      | \ | / | \ | / | \ |
  layer 0 (all):    o-o-o-o-o-o-o-o-o-o-o-o-o    <- every vector, short links

  search: start at the top, greedily hop to the closest neighbor,
          drop a layer, repeat -> reaches the region of the answer fast.
```
Like an express train (top layers) then local stops (bottom) instead of walking every station.

In [ ]:
# Use hnswlib if available; otherwise explain + fall back to exact.
try:
    import hnswlib
    p = hnswlib.Index(space="l2", dim=dim)
    p.init_index(max_elements=N, ef_construction=200, M=16)
    p.add_items(data, np.arange(N))
    p.set_ef(50)                      # ef = search breadth (higher = more accurate, slower)
    t=time.time()
    labels, _ = p.knn_query(query, k=5)
    print("HNSW top-5:", labels[0], f"  ({(time.time()-t)*1000:.1f} ms)")
    overlap = len(set(labels[0]) & set(exact_top))
    print(f"overlap with exact: {overlap}/5  (recall of the ANN index)")
    engine="hnswlib"
except ImportError:
    print("hnswlib not installed (pip install hnswlib to run the real thing).")
    print("Concept: HNSW would return ~the same top-5 in a fraction of the time at scale.")
    engine="explain-only"
print("engine:", engine)

## 3. The accuracy/speed dial
```
  M               : links per node (graph density)   higher = better recall, more memory
  ef_construction : effort when BUILDING the index   higher = better graph, slower build
  ef (search)     : effort when SEARCHING            higher = better recall, slower query
```
**These are the knobs Chroma/Weaviate expose.** If recall is poor, raise `ef`; if the index is huge/slow to build, lower `ef_construction` or `M`.

**Observe:** HNSW returns nearly the same neighbors as exact search but scales to millions of vectors. The 'A' in ANN means you accept ~1–5% of neighbors being off in exchange for orders-of-magnitude speed — almost always the right trade. This is why your Chroma collections stay fast as they grow.